## Exercise: Comparison of the Estimation Algorithms

In this exercise, the three algorithms to estimate the coefficients of an AR($N$)-model (autocorrelation method, autocovariance method and Burg algorithm) should be compared with respect to their **bias** and **variance**.

In addition, a method is examined which is based on the autocorrelation method and applies a **Hamming window** to assess the data instead of a rectangular window
(MATLAB function: `hamming(·)`).

**Given:** (all-pole) transfer system

$$
H(z) = \frac{1}{A(z)}
$$

with

$$
\begin{aligned}
A(z) =;& 1 - 2.2346 z^{-1} + 2.3921 z^{-2} - 2.0324 z^{-3} \
&+ 1.2865 z^{-4} - 0.4954 z^{-5} + 0.2322 z^{-6} \
&- 0.2554 z^{-7} - 0.1848 z^{-8}
\end{aligned}
$$

of order $m = 8$, which is excited by a stationary, zero mean, white process
$N(k)$ with power

$$
\sigma_{\pi N}^2 = 1 .
$$

<div style="text-align:center">
  <img src="../CDR/l15sys.eps" width="70%">
</div>

---

### Tasks

**a)**
In a first step, calculate the pole-zero plot of the system using the command
`zplane(·)`.

For a more exact analysis of the system, determine the exact PARCOR coefficients

$$
\gamma_i, \quad i = 1 \ldots 8 .
$$

For this, determine the unbiased autocorrelation series $r_{\pi XX}(\kappa)$
using the impulse response of the system and solve the exact Yule–Walker equation (9.3.17) by applying the **Levinson–Durbin recursion**.

---

**b)**
Estimate the power density spectrum of the process $X(k)$ by an AR($8$) process using the methods described above.

For this, use **1000 realizations** of the stochastic process $X(k)$, where each interval of samples should have a length of

$$
N = 64
$$

samples.

To take into account the transient response of the system $H(z)$, use at least

$$
2^{11} + 64
$$

samples of the process $N(k)$.
Use the command `filter(·)` to filter the white noise.

Plot the following results and/or give the calculated values:

* **AR coefficients**:
  exact value $a_i$ / mean value $\bar{a}_i$ / mean power of the error

  $$
  \frac{1}{n} \sum_{i=0}^{n} \lvert \bar{a}_i - a_i \rvert^2
  $$

  standard deviation $\sigma$ as well as the $2\sigma$ boundaries

* **PDS**:
  exact and mean power density spectrum as well as the $2\sigma$ boundaries

Compare the different algorithms with respect to **bias** and **variance** of the estimation.

The following MATLAB functions realize the different algorithms:

* autocorrelation method: `lyw(·)`
* autocovariance method: `lcov(·)`
* Burg algorithm: `lburg(·)`


In [ ]:
# BEGIN SOLUTION
MA = [1];
AR = [1.0000   -2.2346    2.3921   -2.0324,...
      1.2865   -0.4954    0.2322   -0.2554    0.1848];
NFFT = 2^6;
n = length(AR)-1;

N = [64];
N_vor = 2^11;
mc_run = 2e3;

% #####  Part a:  Transient Behavior #####
% exact ACF (steady state)

% # Graphical Output # %
figure;
[h1,h2,h3] = zplane(MA,AR); 
set(h1,'Visible','off');
title(sprintf('Poles of the AR(%d) Model',length(AR)-1)); 
grid;
xlabel('Real Part'); 
ylabel('Imaginary Part');

h_arma = impz(MA,AR);
rxx_all = xcorr(h_arma);
index = (length(rxx_all)+1)/2;
rxx = rxx_all(index:index+n);
var = rxx(1);
g(1) = rxx(2)/rxx(1);
var = [var (1-abs(g(1)).^2)*var];
ARcoeff = [1 -g(1)];

for i = 2:n
    g(i) = (ARcoeff*rxx(i+1:-1:2))./var(i);
    var = [var (1-abs(g(i)).^2)*var(i)];
    ARcoeff = [ARcoeff 0]-fliplr(conj([ARcoeff 0]))*g(i);
end;

disp('Exact PARCOR Coefficients:');
disp(sprintf('%1.4f\n',g));
input('....Press RETURN to continue');

% #####  Part b:  Exact PSD #####
Omega_norm  = 0:1/NFFT:1-1/NFFT;
Sxx_ex = abs(fft(MA,NFFT)./fft(AR,NFFT)).^2;
Sxx_ar = zeros(mc_run,NFFT);
Coeff = zeros(mc_run,length(AR));

disp('.......Please wait: Calculation in progress.....');

for run = 1:4
    for run1 = 1:mc_run
        noise = randn(1,20*N+N_vor);
        x_all = filter(MA,AR,noise);

        switch run
            case 1; 
                x = x_all(N_vor+1:N_vor+1+N);
                [ARcoeff] = lyw(x,n,1); 
                str = 'YW';
            case 2; 
                x = x_all(N_vor+1:N_vor+1+N);
                x_ham = x.*hamming(length(x)).';
                [ARcoeff] = lyw(x_ham,n,1); 
                str = 'YW-Ham.';
            case 3; 
                x = x_all(N_vor+1:N_vor+1+N);
                [ARcoeff] = lcov(x,n); 
                str = 'Covar.';
            case 4; 
                x = x_all(N_vor+1:N_vor+1+N);
                [ARcoeff] = lburg_algo(x,N,n,MA,AR); 
                str = 'Burg';
        end;
        
        Coeff(run1,:) = ARcoeff;
        Sxx_ar(run1,:) = abs(ones(1,NFFT)./fft(ARcoeff,NFFT)).^2;
    end;

    Coeff_mean = mean(Coeff,1);
    Coeff_std = std(Coeff,1);
    C2sig_o = Coeff_mean+2*Coeff_std;
    C2sig_u = Coeff_mean-2*Coeff_std;
    Coeff_lst = mean(abs(AR-Coeff_mean).^2);
    Sxx_mean = mean(Sxx_ar,1);
    Sxx_2sigma = Sxx_mean+2*std(Sxx_ar,1);
    
    figure;
    h1=stem([0:n],AR,'ro');  hold on;
    h2=stem([0:n],Coeff_mean,'bx');  hold on;
    h3=stem([0:n],C2sig_o,'go','filled'); hold on;
    h4=stem([0:n],C2sig_u,'go','filled');
    set(h1,'Linestyle','none');
    set(h2,'Linestyle','none');
    set(h3,'Linestyle','none');
    set(h4,'Linestyle','none')
    x_lines=[[0:n]; [0:n]; nan*ones(1,n+1)];
    stddev_lines=[C2sig_u; C2sig_o; nan*ones(1,n+1)];
    plot(x_lines(:),stddev_lines(:),'g');
    title([sprintf('%s-AR(%d) Coefficients; N=%d',str,n,N)]);
    xlabel('i'); ylabel('Coefficients'); grid; axis([0 length(AR)-1 -4 4]);
    legend([h1(1),h2(1),h3(1)],'Exact',...
        ['Mean./E\{e^2\}=',sprintf('%1.2e',Coeff_lst)],...
        ['2\sigma Bound/ \sigma=',sprintf('%1.2f',mean(Coeff_std))]);
    hold off;

    % # Graphical Output # %
    figure;
    h1 = plot(Omega_norm, 10*log10(Sxx_ex/max(Sxx_ex)),'r--');     hold on;
    h2 = plot(Omega_norm, 10*log10(Sxx_mean/max(Sxx_ex)),'b-');    hold on;
    h3 = plot(Omega_norm, 10*log10(Sxx_2sigma/max(Sxx_ex)),'g-.'); hold off;
    ylabel('S_{xx}(e^{j\Omega}) in dB');
    legend([h1,h2,h3],'Exact','Mean','2\sigma Bound');
    title(sprintf('PSD of %s-AR(%d); N=%d',str,n,N));  
    xlabel('\Omega/2\pi');
    grid;
end;
# END SOLUTION

## Solution: Comparison of the Estimation Algorithms

### a)

The figure shows the pole-zero plot of the system $H(z)$. It is a real-valued system, where a pole pair at

$$
\Omega / 2\pi \approx 0.05
$$

is located very close to the unit circle.

For the underlying AR(8) process, the following PARCOR coefficients result when the exact Yule–Walker equation is used:

$$
\gamma_1 = 0.9785,;
\gamma_2 = -0.9279,;
\gamma_3 = 0.1531,;
\gamma_4 = -0.6974,
$$

$$
\gamma_5 = 0.1815,;
\gamma_6 = -0.1563,;
\gamma_7 = -0.1631,;
\gamma_8 = -0.1848 .
$$

Note that the first two PARCOR coefficients have values close to one and therefore dominate the behavior of the AR process. This AR process corresponds to a typical speech process.

<div style="text-align:center">
  <img src="../pngs/ldsv9a04_a1.png">
</div>

---

### b)

In the following, the essential properties of the known spectral estimation algorithms are summarized.
A sample interval of length $N$ of the process $X(k)$ is assumed. For the discussion, the **power of the prediction error** is considered.
Because of the inverse relationship between an AR($n$) model and a predictor, minimizing the prediction error power is equivalent to determining the parameters of an AR($n$) process. In other words, the process $X(k)$ is assigned a predictor whose coefficients are chosen such that the prediction error power is minimized and equals the underlying white noise process $N(k)$.

---

#### Yule–Walker / Autocorrelation Method

* Process $X(k)$ is assumed to vanish outside the sample interval (rectangular window)
* The setting time of the predictor is included in the minimization
* The **biased** autocorrelation sequence $\hat{r}_{\pi XX}(\kappa)$ is used
* The solution is based on a Hermitian Toeplitz matrix
* The estimation is **biased**, but the AR model is always **minimal phase**

---

#### Autocovariance Method

* Process $X(k)$ is assumed to be undefined outside the sample interval
* The setting time of the predictor is **not** included in the minimization
* The **unbiased** autocorrelation sequence $r_{\pi XX}(\kappa)$ is used
* No Toeplitz matrix exists
* The estimation is **unbiased**, but the AR model is not necessarily **minimal phase**

---

#### Burg Algorithm

* Joint minimization of forward and backward prediction error powers
* Process $X(k)$ is treated as undefined outside the sample interval
* The setting time of the predictor is included in the minimization
* The estimation is **unbiased** and the AR model is always **minimal phase**

---

#### Autocorrelation Method with Hamming Window

Instead of applying a rectangular window to the process $X(k)$, a Hamming window of the same length $N$ is applied.

Rather than using the correlation of two rectangular windows—which results in a Bartlett window—to compute the biased autocorrelation sequence, the correlation of two Hamming windows is used. This introduces a small bias for autocorrelation values $\hat{r}_{\pi XX}(\kappa)$ at small $\kappa$, while larger $\kappa$ values receive stronger weighting.

To illustrate this behavior, the unbiased correlation results of both windows are plotted. This method is frequently used in speech signal analysis, since speech signals typically exhibit a dominant first PARCOR coefficient, which is mainly determined by $\hat{r}*{\pi XX}(0)$ and $\hat{r}*{\pi XX}(1)$.

<div style="text-align:center">
  <img src="../pngs/ldsv9a04fenster.png">
</div>

---

The figures on the left-hand side show the estimated coefficients, while the figures on the right-hand side show the corresponding power density spectra.

<div style="text-align:center">
  <img src="../pngs/ldsv9a04_b11.png">
  <img src="../pngs/ldsv9a04_b12.png">
</div>

The results based on the autocorrelation method (“YW-AR(8)”) confirm the presence of bias in this estimation. For example, some exact coefficients such as $a_3$ lie outside the $2\sigma$ boundaries. This behavior is also reflected in the relatively high prediction error power

$$
E{e^2} = 1.18 .
$$

The power density spectrum is poorly approximated, and the critical pole at $\Omega / 2\pi \approx 0.05$ is incorrectly modeled.

---

<div style="text-align:center">
  <img src="../pngs/ldsv9a04_b21.png">
  <img src="../pngs/ldsv9a04_b22.png">
</div>

Using the autocorrelation method modified with a Hamming window (“YW-HAM.-AR(8)”), a significant reduction of bias is observed for processes with large PARCOR coefficients $\gamma_1$ and $\gamma_2$.

For instance, coefficient $a_3$, which previously lay outside the $2\sigma$ boundaries, now lies inside them. The mean prediction error power decreases to

$$
E{e^2} = 0.25 .
$$

A drawback of the Hamming weighting is an increased standard deviation $\sigma$.
The estimated power density spectrum shows a clear improvement when the Hamming window is used.

---

<div style="text-align:center">
  <img src="../pngs/ldsv9a04_b31.png">
  <img src="../pngs/ldsv9a04_b32.png">
</div>

An unbiased estimation of the coefficients is achieved using the covariance method (“cov.-AR(8)”).
The small deviations between the exact and mean values are due to the finite number of realizations ($1000$). The standard deviation behaves similarly to the modified autocorrelation method.

Thus, the covariance method represents an attractive alternative for spectral estimation, as confirmed by the approximation of the power density spectrum. Its main drawback is the lack of guaranteed minimal-phase behavior.

---

<div style="text-align:center">
  <img src="../pngs/ldsv9a04_b41.png">
  <img src="../pngs/ldsv9a04_b42.png">
</div>

The figures above show the results obtained with the Burg algorithm (“Burg-AR(8)”).
The joint minimization of forward and backward prediction errors proves to be the most suitable approach. The absence of bias is clearly visible, and the power density spectrum is approximated almost as well as with the covariance method.

Regarding estimation variance, the Burg algorithm behaves similarly to the covariance and modified autocorrelation methods for the example considered here.
